In [ ]:
import matplotlib.pyplot as plt
import multiprocessing
import seaborn as sns
import json
from dataset_utils import *

In [ ]:
from huggingface_hub import snapshot_download
import os

# The dataset is public; no token needed. Optionally `export HF_TOKEN=...`
# in your shell for higher rate limits / private datasets.
cache_dir = "./hf_cache"
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir

def download_repo(repo_id, repo_type="dataset", cache_dir="./hf_cache"):
    """Download repository files to local cache."""
    if 'HF_HOME' in os.environ:
        cache_dir = os.environ['HF_HOME']
    else:
        print("Using default cache directory:", cache_dir)
    try:
        local_dir = snapshot_download(
            repo_id=repo_id,
            repo_type=repo_type,
            cache_dir=cache_dir
        )
        return local_dir
    except Exception as e:
        print(f"Error downloading: {e}")
        return None


In [ ]:
dataset_name = "lbnl-metabolomics/20210915_JGI-AK_MK_506588_SoilWaterRep_final_QE-HF_C18_USDAY63680"

# Check repository (purely diagnostic; confirms the repo exists)
repo_type, files = check_repo(dataset_name)

# Download if found
local_dir = download_repo(dataset_name)

my_files = get_paths_downloaded_files(local_dir)

In [ ]:


metadata = []

for filename in my_files:
    # print(f"Processing file: {filename}")
    encoder = SpectraCodec()
    # print(f"Decoding file: {filename}")
    # try:
    decoded_message= encoder.decode_message_from_file(filename, method='hilbert')
    decoded_message = json.loads(decoded_message)
    # decoded_message = eval(decoded_message)
    decoded_message['filename'] = filename
    metadata.append(decoded_message)
    # except Exception as e:
        # print(f"Error processing {filename}: {e}")
        # continue
metadata = pd.DataFrame(metadata)
metadata.head()

In [ ]:
my_attribute = 'InternalStandardsUsed'
my_file = '20210915_JGI-AK_MK_506588_SoilWaterRep_final_QE-HF_C18_USDAY63680_POS_MSMS_73_ExCtrl_A_Rg80to1200-CE102040-soil-S1_Run14.mzML'
idx = metadata['filename'].str.contains(my_file)
print(f"Metadata for file: {my_file}")
my_value = metadata.loc[idx, my_attribute].values
print(f"Metadata for {my_attribute}: {my_value}")
my_attribute = 'IonizationSourceAndPolarity'
my_value = metadata.loc[idx, my_attribute].values
print(f"Metadata for {my_attribute}: {my_value}")


In [ ]:
abmba_mz = 228.97384+1.007276
mz_tolerance = 0.01  # ppm tolerance

# Find files with ABMBA internal standard in positive ionization mode
idx1 = metadata['IonizationSourceAndPolarity'].str.contains('positive',case=False)
idx2 = metadata['InternalStandardsUsed'].str.contains('abmba',case=False)
idx = idx1 & idx2
# Get the filenames of the files to analyze
files_to_analyze = metadata.loc[idx, 'filename'].values
# Prepare the list of files to analyze with target m/z and tolerance
files_to_analyze = [{'filename': f, 'target_mz':abmba_mz,'mz_tolerance':mz_tolerance} for f in files_to_analyze]
files_to_analyze[:3]

In [ ]:
eic = get_eic(files_to_analyze[0])
eic_df = pd.DataFrame(eic, columns=['scan_time', 'intensity', 'mz', 'filename'])
eic_df.head()

In [ ]:

num_processes = multiprocessing.cpu_count()
num_processes = 8  # Set to 4 for testing, change as needed
print(f"Starting EIC extraction with {num_processes} processes...")
with multiprocessing.Pool(processes=num_processes) as pool:
    eic_df = pool.map(get_eic, files_to_analyze)
eic_df = [item for sublist in eic_df for item in sublist]  # Flatten the list
eic_df = pd.DataFrame(eic_df, columns=['scan_time', 'intensity', 'mz', 'filename'])
eic_df['rt'] = eic_df['scan_time'].apply(lambda x: x[0])
eic_df.head()

In [ ]:
abmba_eic_grouped = eic_df.groupby('filename').agg({'rt': list, 'intensity': list, 'mz': list}).reset_index()
abmba_eic_grouped

In [ ]:
abmba_eic_grouped = eic_df.groupby('filename').agg({'rt': list, 'intensity': list, 'mz': list}).reset_index()
# convert to numpy arrays for easier manipulation
abmba_eic_grouped['rt'] = abmba_eic_grouped['rt'].apply(lambda x: np.array(x))
abmba_eic_grouped['intensity'] = abmba_eic_grouped['intensity'].apply(lambda x: np.array(x))
abmba_eic_grouped['mz'] = abmba_eic_grouped['mz'].apply(lambda x: np.array(x))

# smooth the intensity using a moving average
# def smooth_intensity(intensity, window_size=4):
    # return np.convolve(intensity, np.ones(window_size)/window_size, mode='same')
# abmba_eic_grouped['intensity'] = abmba_eic_grouped['intensity'].apply(smooth_intensity)

abmba_eic_grouped['rt_peak_index'] = abmba_eic_grouped['intensity'].apply(lambda x: np.argmax(x))
abmba_eic_grouped['rt_peak'] = abmba_eic_grouped.apply(lambda row: row['rt'][row['rt_peak_index']], axis=1)
abmba_eic_grouped['intensity_peak'] = abmba_eic_grouped.apply(lambda row: row['intensity'][row['rt_peak_index']], axis=1)
abmba_eic_grouped['mz_peak'] = abmba_eic_grouped.apply(lambda row: row['mz'][row['rt_peak_index']], axis=1)
abmba_eic_grouped['mz_centroid'] = abmba_eic_grouped.apply(lambda row: sum(row['mz']* row['intensity']) / sum(row['intensity']), axis=1)
abmba_eic_grouped['mz_error'] = abmba_eic_grouped['mz_centroid'].apply(lambda x: x - abmba_mz)
abmba_eic_grouped['mz_error_ppm'] = abmba_eic_grouped['mz_error'] / abmba_mz * 1e6
median_rt = abmba_eic_grouped['rt_peak'].median()
abmba_eic_grouped['rt_error'] = abmba_eic_grouped['rt_peak'].apply(lambda x: x - median_rt)
abmba_eic_grouped

In [ ]:
import seaborn as sns
fig,(ax1,ax2,ax3) = plt.subplots(figsize=(14, 6),ncols=3, nrows=1)
for _, row in abmba_eic_grouped.iterrows():
    ax1.plot(row['rt'], row['intensity'], label=row['filename'])
ax1.set_xlabel('Retention Time (min)', fontsize=18)
ax1.set_ylabel('Intensity', fontsize=18)
# ax1.set_yscale('log')
# ax1.set_xlim(4.5,5)
ax1.set_xlim(median_rt - 0.1, median_rt + 0.1)

ax1.text(0.05, 0.95, 'a', transform=ax1.transAxes, fontsize=33, verticalalignment='top')

data = abmba_eic_grouped['mz_error_ppm']
y2= ax2.hist(data,weights=np.ones_like(data)/len(data), bins=15,density=False,alpha=0.5)
ax2.set_xlabel('m/z Error (ppm)', fontsize=18)
ax2.set_ylabel('Frequency', fontsize=18)
ax2.text(0.05, 0.95, 'b', transform=ax2.transAxes, fontsize=33, verticalalignment='top')
ax2.set_xlim(-1.5, 1.5)

data = abmba_eic_grouped['rt_error']
y = ax3.hist(data, weights=np.ones_like(data)/len(data),
             bins=15,density=False,alpha=0.5)
ax3.set_xlabel('RT Error (min)', fontsize=18)
ax3.set_ylabel('Frequency', fontsize=18)
ax3.text(0.05, 0.95, 'c', transform=ax3.transAxes, fontsize=33, verticalalignment='top')
ax3.set_xlim(-0.02, 0.02)
# make tick fonts larger
for ax in (ax1, ax2, ax3):
    ax.tick_params(axis='both', which='major', labelsize=16)
    ax.tick_params(axis='both', which='minor', labelsize=12)
    ax.yaxis.get_offset_text().set_fontsize(16) # Adjust size as needed

# remove top and right spines
for ax in (ax1, ax2, ax3):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
y[0].sum(),y2[0].sum()

In [ ]:
# AMINOADIPATE
# 3-Aminosalicylic acid
# DL-2-Methylglutamic acid
# L-Citrulline
# Geniposide
# Lyngbyatoxin A
# Deoxyfructosazine
# .delta.-Hexalactone
metabolites = {
    # "Ectoine": {
    #     "formula": "C6H10N2O2",
    #     "monoisotopic_weight": 142.0742,
    #     "pubchem_cid": 123162,
    #     "smiles": "C[N+]1(C)CC(NC(=O)C1)C(=O)O"
    # },
    "Delta-hexalactone": {
        "formula": "C6H10O2",
        "monoisotopic_weight": 114.0681,
        "pubchem_cid": 12553,
        "smiles": "CCCCC1CC(=O)O1"
    },
    # "Phthalic acid": {  # The parent compound of phthalic acid derivatives
    #     "formula": "C8H6O4",
    #     "monoisotopic_weight": 166.0266,
    #     "pubchem_cid": 1017,
    #     "smiles": "c1ccc2c(c1)C(=O)OC2=O"
    # },
    # "3-methyladipic acid": {
    #     "formula": "C7H12O4",
    #     "monoisotopic_weight": 160.0736,
    #     "pubchem_cid": 192843,
    #     "smiles": "CC(CCC(=O)O)CC(=O)O"
    # },
    # "Neoabietic acid": {
    #     "formula": "C20H30O2",
    #     "monoisotopic_weight": 302.2246,
    #     "pubchem_cid": 94208,
    #     "smiles": "CC(C)C1=CC2=CCC3C(C)(C(=O)O)CCCC3(C)C2CC1"
    # },
    # "Sulfosalicylic acid": {
    #     "formula": "C7H6O6S",
    #     "monoisotopic_weight": 217.9885,
    #     "pubchem_cid": 2723,
    #     "smiles": "c1cc(c(cc1S(=O)(=O)O)C(=O)O)O"
    # },

    "3-Aminosalicylic acid": {
        "formula": "C7H7NO3",
        "monoisotopic_weight": 153.0426,
        "pubchem_cid": 6070,
        "smiles": "C1=CC(=C(C=C1)N)C(=O)O"
    },
    "DL-2-Methylglutamic acid": {
        "formula": "C6H11NO4",
        "monoisotopic_weight": 147.0681,
        "pubchem_cid": 5284560,
        "smiles": "CC(C(=O)O)C(C(=O)O)N"
    },
    "L-Citrulline": {
        "formula": "C6H13N3O3",
        "monoisotopic_weight": 175.0894,
        "pubchem_cid": 5957,
        "smiles": "C(C(=O)O)C(N)C(=O)N"
    },
    "Geniposide": {
        "formula": "C17H24O10",
        "monoisotopic_weight": 392.1312,
        "pubchem_cid": 5280800,
        "smiles": "CC1=CC(=C(C=C1)O)C(=O)OCC2C(C(C(C(O2)CO)O)O)O"
    },
    "Lyngbyatoxin A": {
        "formula": "C20H30N2O5",
        "monoisotopic_weight": 374.2121,
        "pubchem_cid": 5280800,
        "smiles": "CC1=CC(=C(C=C1)O)C(=O)OCC2C(C(C(C(O2)CO)O)O)O"
    },
    "Deoxyfructosazine": {
        "formula": "C12H21NO5",
        "monoisotopic_weight": 257.1392,
        "pubchem_cid": 5280800,
        "smiles": "CC1=CC(=C(C=C1)O)C(=O)OCC2C(C(C(C(O2)CO)O)O)O"
    },
    # "Coumaric acid": {
    #     "formula": "C9H8O3",
    #     "monoisotopic_weight": 164.0526,
    #     "pubchem_cid": 6371,
    #     "smiles": "C1=CC=C(C=C1)C(=O)OC2=CC=CC=C2"
    # },
    # "arginine": {
    #     "formula": "C6H14N4O2",
    #     "monoisotopic_weight": 174.1117,
    #     "pubchem_cid": 147,
    #     "smiles": "C[C@@H](N)C(=O)OCC(N)=N"
    # },
}
metabolites_df = pd.DataFrame(metabolites).T
metabolites_df['pos_mz'] = metabolites_df.apply(lambda x: x['monoisotopic_weight'] + 1.007276 if not '+' in x['smiles'] else x['monoisotopic_weight'], axis=1)
metabolites_df.index.name = 'metabolite_name'
metabolites_df.reset_index(inplace=True)
metabolites_df

In [ ]:
from importlib import reload
import dataset_utils
reload(dataset_utils)


data = dataset_utils.get_ms1_data(files_to_analyze[0]['filename'])

feature_df = dataset_utils.get_peak_height(data, metabolites_df)

feature_df

In [ ]:
from importlib import reload
import dataset_utils
reload(dataset_utils)

import multiprocessing
from functools import partial
import pandas as pd

# Create a worker function where the `metabolites_df` argument is pre-filled.
# This is the correct and efficient way to pass this large, constant DataFrame.
worker_func = partial(dataset_utils.get_compound_data_from_files, metabolites_df=metabolites_df)

# The `if __name__ == '__main__':` guard is crucial for multiprocessing to work reliably.
if __name__ == '__main__':
    num_processes = 8#multiprocessing.cpu_count()
    # We are processing the first 8 files as in your original code.
    files_to_process = [f['filename'] for f in files_to_analyze]
    print(f"Starting feature extraction on {len(files_to_process)} files with {num_processes} processes...")

    with multiprocessing.Pool(processes=num_processes) as pool:
        # pool.map passes each item from `files_to_process` as the first argument
        # to `worker_func`.
        list_of_dfs = pool.map(worker_func, files_to_process)

    # Combine the results into a single DataFrame
    if list_of_dfs:
        feature_dfs = pd.concat(list_of_dfs, ignore_index=True)
        print("Processing complete.")
        print(feature_dfs.head())
    else:
        print("Warning: Processing returned no data.")
        feature_dfs = pd.DataFrame()

In [ ]:
feature_dfs

In [ ]:
feature_dfs['basename'] = feature_dfs['filename'].apply(lambda x: os.path.basename(x))
feature_dfs['sample_type'] = feature_dfs['basename'].apply(lambda x: x.split('_')[12])
feature_dfs['sample'] = feature_dfs['sample_type'].apply(lambda x: x.split('-')[0])
feature_dfs['time'] = feature_dfs['sample_type'].apply(lambda x: x.split('-')[1] if '-' in x else 'unknown')
metadata = [
    # Converting the existing data structure with additional fields
    {
        'basename': item['basename'],
        # 'sample': item['sample'],
        # 'time': item['time'],
        'timepoint_days': int(item['time'][1:]) if item['time'].startswith('D') else None,
        'replicate': item['basename'].split('_')[-1].split('-')[0] if 'Run' not in item['basename'].split('_')[-1] else item['basename'].split('-')[-2].split('_')[-1],
        'phenotype': 'Hydrophilic' if item['sample'] in ['S16', 'S17', 'S32'] else 
                    'Hydrophobic' if item['sample'] in ['S29', 'S40', 'S53'] else 
                    'Control',
        'control_type': 'Negative Control' if item['sample'] == 'Neg' else
                       'Sterile Component' if item['sample'] == 'Sterile' else
                       'Extraction Control' if item['sample'] == 'ExCtrl' else None,
        'instrument': 'QE-HF',
        'column': 'C18',
        'ionization_mode': 'Positive',
        'analysis_type': 'MSMS' 
    }
    for item in feature_dfs.to_dict(orient='records')
]
metadata = pd.DataFrame(metadata)
metadata.drop_duplicates(inplace=True)
metadata

In [ ]:
cols =['basename','sample','time']
print(feature_dfs[cols].drop_duplicates().to_dict(orient='records').__str__())

In [ ]:
p = pd.pivot_table(feature_dfs, index='basename', columns='metabolite_name', values='area', aggfunc='sum', fill_value=0)

p.reset_index(inplace=True)
p = pd.merge(p, metadata, on='basename', how='left')
p


In [ ]:
sys.maxunicode

In [ ]:
chr(1114111)

In [ ]:
[ord(c) for c in ['⏣','⌬']]

In [ ]:
compounds = feature_dfs['metabolite_name'].unique()
fig,ax = plt.subplots(figsize=(14, 6),nrows=2,ncols=np.ceil(len(compounds)/2).astype(int), sharex=True)
ax = ax.flatten()
for i, compound in enumerate(compounds):
    idx = p['control_type'] != 'None'
    sns.boxplot(data=p[idx], x='timepoint_days', y=compound, hue='phenotype', ax=ax[i], palette='Set2', showfliers=False)
    ax[i].set_title(compound)
    ax[i].set_xlabel('Time (days)')
    ax[i].set_ylabel('Integrated Peak Area')
    # disable legend for all but the last subplot
    
    ax[i].get_legend().remove()
    

# move legend to the outside of the last subplot
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.15), fontsize=14)
# adjust layout
plt.tight_layout()
plt.show()
